## Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect 
signatures of anomalous or strongly anomalous diffusion. The analysis 
is performed first on the full aggregated dataset, then compared across 
distance groups. For each signal, the $q$-th order moment is computed 
at different time scales $\tau$:

$$M_q(\tau) = \langle |a|^q \rangle_\tau$$

If the process exhibits scaling, the moments obey a power law:

$$M_q(\tau) \sim \tau^{\zeta(q)}$$

For normal diffusion, $\zeta(q) = q/2$ (linear in $q$). Deviations from 
linearity indicate anomalous or strongly anomalous diffusion.

In [ ]:
from pathlib import Path
import pandas as pd
import logging
from src.signals_scaling import (compute_moment_scaling_acc,compute_moment_scaling_vel, compute_moment_scaling_disp,
                         compute_scaling_exponents,test_scaling_linearity, fit_piecewise_scaling,
                         trim_to_event_window, compute_increment_tail_exponents)
from src.plot_settings import set_plot_style
from src.plots import plot_onset_diagnostic, plot_onset_distribution, plot_increment_distributions, plot_r2_diagnostic
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

### Computation of q-th order moments

Moments are computed for moment orders $q \in \{0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0\}$ and time scales $\tau \in \{10, 50, 100, 200, 500, 1000, 2000, 5000, 10000\}$ samples.

In [ ]:
q_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Compute moments — global signal
df_moments_agg = compute_moment_scaling(df_global, q_values, tau_values)
print("Shape:", df_moments_agg.shape)

try:
    df_moments_agg.to_parquet('../data/processed/moments_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

A plot of $\zeta(q)$ vs $q$ is produced for each signal, with the reference 
line $\zeta(q) = q/2$ corresponding to normal diffusion.

In [ ]:
# Compute scaling exponents on full aggregated dataset
df_exponents_agg = compute_scaling_exponents(df_moments_agg,
                                              output_dir=FIGURES_DIR / 'scaling')
try:
    df_exponents_agg.to_parquet('../data/processed/scaling_exponents_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing a linear 
and a quadratic fit using AIC. A piecewise linear fit is also performed 
to detect the presence of two distinct scaling regimes:

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes.

In [ ]:
# Linearity check on full aggregated dataset
df_linearity_agg = test_scaling_linearity(df_exponents_agg,
                                           output_dir=FIGURES_DIR / 'scaling')
try:
    df_linearity_agg.to_parquet('../data/processed/scaling_linearity_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

# Piecewise linear fit on full aggregated dataset
df_piecewise_agg = fit_piecewise_scaling(df_exponents_agg,
                                          output_dir=FIGURES_DIR / 'scaling')
try:
    df_piecewise_agg.to_parquet('../data/processed/scaling_piecewise_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## 7. Summary

A summary table collects the main results from all analyses, first for 
the full aggregated dataset, then for each distance group. The table 
allows a direct comparison of the statistical properties of the signals 
across distance groups.

| Column | Description |
|--------|-------------|
| `kurtosis` | Excess kurtosis $\kappa$ |
| `skewness` | Skewness $\gamma$ |
| `non_gaussian` | Anderson-Darling test result ($\alpha = 0.05$) |
| `best_fit_aic` | Best fitting distribution by AIC |
| `student_t_df` | Student-t degrees of freedom $\nu$ |
| `power_law_exp` | Power law exponent $\hat{\alpha}$ (Hill estimator) |
| `q_break` | Piecewise scaling breakpoint $q^*$ |
| `slope_low` | Scaling slope for $q \leq q^*$ |
| `slope_high` | Scaling slope for $q > q^*$ |
| `beta` | Autocorrelation scaling exponent $\beta$ |

In [ ]:
# Summary for full aggregated dataset
df_summary_agg = df_gaussian_results_agg[['station', 'stream', 'kurtosis', 'skewness', 'non_gaussian']]\
    .merge(df_heavy_tail_agg[['station', 'stream', 'best_fit_aic', 'student_t_df', 'power_law_exp']],
           on=['station', 'stream'])\
    .merge(df_piecewise_agg[['station', 'stream', 'q_break', 'slope_low', 'slope_high']],
           on=['station', 'stream'])\
    .merge(df_autocorr_scaling_agg[['station', 'stream', 'beta', 'r2']],
           on=['station', 'stream'])

print("Summary — full aggregated dataset:")
pd.set_option('display.max_rows', None)
display(df_summary_agg)
pd.reset_option('display.max_rows')

try:
    df_summary_agg.to_parquet('../data/processed/summary_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

# Summary by distance group
df_summary_by_group = df_gaussian_by_group[['group', 'kurtosis', 'skewness', 'non_gaussian']]\
    .merge(df_heavy_tail_by_group[['group', 'best_fit', 'student_t_df', 'power_law_exp']],
           on='group')\
    .merge(
        pd.DataFrame([{
            'group': group,
            'q_break': df_piecewise_by_group[group]['q_break'].mean(),
            'slope_low': df_piecewise_by_group[group]['slope_low'].mean(),
            'slope_high': df_piecewise_by_group[group]['slope_high'].mean(),
            'beta': df_autocorr_scaling_by_group[group]['beta'].mean(),
        } for group in groups]),
        on='group'
    )

print("\nSummary — by distance group:")
display(df_summary_by_group)

try:
    df_summary_by_group.to_parquet('../data/processed/summary_by_group.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")